# Завдання 1

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [1]:
import urllib.request
import os
import pandas as pd
from datetime import datetime

In [2]:
def download_vhi():
    path = "vhi_data"
    
    if not os.path.exists(path):
        os.makedirs(path)

    for i in range(1, 28):
        if any(f.startswith(f"vhi_id_{i}_") for f in os.listdir(path)):
            print(f"Область №{i} вже є в папці. Пропускаємо.")
            continue

        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        
        now = datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"vhi_id_{i}_{now}.csv"
        filepath = os.path.join(path, filename)

        try:
            print(f"Завантаження області №{i}...")
            urllib.request.urlretrieve(url, filepath)
            print(f"Збережено: {filename}")
        except Exception as e:
            print(f"Не вдалося завантажити область №{i}: {e}")

In [3]:
if __name__ == "__main__":
    download_vhi()

Область №1 вже є в папці. Пропускаємо.
Область №2 вже є в папці. Пропускаємо.
Область №3 вже є в папці. Пропускаємо.
Область №4 вже є в папці. Пропускаємо.
Область №5 вже є в папці. Пропускаємо.
Область №6 вже є в папці. Пропускаємо.
Область №7 вже є в папці. Пропускаємо.
Область №8 вже є в папці. Пропускаємо.
Область №9 вже є в папці. Пропускаємо.
Область №10 вже є в папці. Пропускаємо.
Область №11 вже є в папці. Пропускаємо.
Область №12 вже є в папці. Пропускаємо.
Область №13 вже є в папці. Пропускаємо.
Область №14 вже є в папці. Пропускаємо.
Область №15 вже є в папці. Пропускаємо.
Область №16 вже є в папці. Пропускаємо.
Область №17 вже є в папці. Пропускаємо.
Область №18 вже є в папці. Пропускаємо.
Область №19 вже є в папці. Пропускаємо.
Область №20 вже є в папці. Пропускаємо.
Область №21 вже є в папці. Пропускаємо.
Область №22 вже є в папці. Пропускаємо.
Область №23 вже є в папці. Пропускаємо.
Область №24 вже є в папці. Пропускаємо.
Область №25 вже є в папці. Пропускаємо.
Область №

# Завдання 2

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [4]:
def load_and_clean_data(folder_path):
    all_data = []
    
    provinces_names = {
        1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea", 5: "Dnipropetrovsk",
        6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyy",
        11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk", 15: "Lviv",
        16: "Mykolayiv", 17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sevastopol",
        21: "Sumy", 22: "Ternopil", 23: "Transcarpathia", 24: "Vinnytsya", 25: "Volyn",
        26: "Zaporizhzhya", 27: "Zhytomyr"
    }

    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    for file in files:
        province_id = int(file.split('_')[2])
        filepath = os.path.join(folder_path, file)
        
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(filepath, skiprows=2, names=headers, skipinitialspace=True)
        
        df = df.drop(columns=['empty'])
        df['Year'] = df['Year'].astype(str).str.replace('<br>', '').str.replace('<pre>', '')
        for col in ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna()
        df['Province_ID'] = province_id
        df['Province_Name'] = provinces_names.get(province_id, "Unknown")
        
        all_data.append(df)

    final_df = pd.concat(all_data, ignore_index=True)
    return final_df

In [5]:
df = load_and_clean_data("vhi_data")
print(df.head())

     Year  Week    SMN     SMT    VCI    TCI    VHI  Province_ID Province_Name
0  1982.0   2.0  0.063  261.53  55.89  38.20  47.04           10  Khmelnytskyy
1  1982.0   3.0  0.063  263.45  57.30  32.69  44.99           10  Khmelnytskyy
2  1982.0   4.0  0.061  265.10  53.96  28.62  41.29           10  Khmelnytskyy
3  1982.0   5.0  0.058  266.42  46.87  28.57  37.72           10  Khmelnytskyy
4  1982.0   6.0  0.056  267.47  39.55  30.27  34.91           10  Khmelnytskyy


# Завдання 3

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька)

In [6]:
import pandas as pd
import os

def load_and_reindex_data(folder_path):
    all_data = []
    noaa_indices = {
        1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea", 5: "Dnipropetrovsk",
        6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyy",
        11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk", 15: "Lviv",
        16: "Mykolayiv", 17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sevastopol",
        21: "Sumy", 22: "Ternopil", 23: "Transcarpathia", 24: "Vinnytsya", 25: "Volyn",
        26: "Zaporizhzhya", 27: "Zhytomyr"
    }
    ua_alphabet_map = {
        "Vinnytsya": "Вінницька",
        "Volyn": "Волинська",
        "Dnipropetrovsk": "Дніпропетровська",
        "Donetsk": "Донецька",
        "Zhytomyr": "Житомирська",
        "Transcarpathia": "Закарпатська",
        "Zaporizhzhya": "Запорізька",
        "Ivano-Frankivsk": "Івано-Франківська",
        "Kyiv City": "Київ",
        "Kyiv": "Київська",
        "Kirovohrad": "Кіровоградська",
        "Crimea": "Крим",
        "Luhansk": "Луганська",
        "Lviv": "Львівська",
        "Mykolayiv": "Миколаївська",
        "Odessa": "Одеська",
        "Poltava": "Полтавська",
        "Rivne": "Рівненська",
        "Sevastopol": "Севастополь",
        "Sumy": "Сумська",
        "Ternopil": "Тернопільська",
        "Kharkiv": "Харківська",
        "Kherson": "Херсонська",
        "Khmelnytskyy": "Хмельницька",
        "Cherkasy": "Черкаська",
        "Chernivtsi": "Чернівецька",
        "Chernihiv": "Чернігівська"
    }

    new_id_map = {name: i + 1 for i, name in enumerate(ua_alphabet_map.values())}

    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    for file in files:
        old_id = int(file.split('_')[2])
        filepath = os.path.join(folder_path, file)
        
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(filepath, skiprows=2, names=headers, skipinitialspace=True)
        df = df.drop(columns=['empty'])
        
        df['Year'] = df['Year'].astype(str).str.replace('<br>', '').str.replace('<pre>', '')
        for col in ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna()

        eng_name = noaa_indices.get(old_id)
        ua_name = ua_alphabet_map.get(eng_name)
        
        df['Province_Name'] = ua_name
        df['Province_ID'] = new_id_map.get(ua_name)
        
        all_data.append(df)

    return pd.concat(all_data, ignore_index=True)

df = load_and_reindex_data("vhi_data")

In [7]:
df = load_and_reindex_data("vhi_data")
print(df.head())

     Year  Week    SMN     SMT    VCI    TCI    VHI Province_Name  Province_ID
0  1982.0   2.0  0.063  261.53  55.89  38.20  47.04   Хмельницька           24
1  1982.0   3.0  0.063  263.45  57.30  32.69  44.99   Хмельницька           24
2  1982.0   4.0  0.061  265.10  53.96  28.62  41.29   Хмельницька           24
3  1982.0   5.0  0.058  266.42  46.87  28.57  37.72   Хмельницька           24
4  1982.0   6.0  0.056  267.47  39.55  30.27  34.91   Хмельницька           24


# Завдання 4

## Ряд VHI для області за вказаний рік

In [8]:
def get_vhi_by_year(df, province_id, year):
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    
    province_name = df[df['Province_ID'] == province_id]['Province_Name'].iloc[0]
    print(f"VHI для області: {province_name} (ID: {province_id}) за {year} рік")
    return result

get_vhi_by_year(df, 5, 2015)

VHI для області: Житомирська (ID: 5) за 2015 рік


,Week,VHI
41945,1.0,49.69
41946,2.0,52.15
41947,3.0,52.27
41948,4.0,50.81
41949,5.0,48.04
41950,6.0,46.59
41951,7.0,45.59
41952,8.0,45.53
41953,9.0,46.09
41954,10.0,46.03


## Ряд VHI за вказаний діапазон років для вказаних областей

In [9]:
def get_vhi_range(df, province_id, start_year, end_year):
    result = df[(df['Province_ID'] == province_id) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)][['Year', 'Week', 'VHI']]
    
    province_name = df[df['Province_ID'] == province_id]['Province_Name'].iloc[0]
    print(f"--- Ряд VHI для області: {province_name} з {start_year} по {end_year} роки ---")
    return result


get_vhi_range(df, 10, 2010, 2012)

--- Ряд VHI для області: Київська з 2010 по 2012 роки ---


,Year,Week,VHI
3690,2010.0,1.0,47.84
3691,2010.0,2.0,47.23
3692,2010.0,3.0,49.70
3693,2010.0,4.0,52.06
3694,2010.0,5.0,52.79
...,...,...,...
3841,2012.0,48.0,48.56
3842,2012.0,49.0,49.98
3843,2012.0,50.0,51.21
3844,2012.0,51.0,49.17


## Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [10]:
def get_vhi_stats(df, province_id, start_year, end_year):
    subset = df[(df['Province_ID'] == province_id) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]['VHI']
    
    province_name = df[df['Province_ID'] == province_id]['Province_Name'].iloc[0]
    
    print(f"Статистика VHI для області: {province_name} ({start_year}-{end_year})")
    stats = {
        "Мінімум": subset.min(),
        "Максимум": subset.max(),
        "Середнє": subset.mean(),
        "Медіана": subset.median()
    }
    
    return pd.Series(stats)

get_vhi_stats(df, 17, 2012, 2024)

Статистика VHI для області: Полтавська (2012-2024)


Мінімум     22.220000
Максимум    80.550000
Середнє     46.845266
Медіана     45.410000
dtype: float64